# ML-05 — Feature Vector and Leakage/Privacy Check

[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/AdarshIsaac/NewRepoML/blob/main/work/notebooks/w03_feature_leakage_check.ipynb?flush_cache=true)

This skeleton is yours to fill. Work the sections **in order** — each one has a one-line hint. Simple words, honest numbers.

> Working with an AI assistant? Tell it to read `skills/README.md` first and load the one skill this assignment names on its card.

## 1. Build the feature vector

The ranking lane needs a page-level vector available before review. I use five safe signals from the starter slice, add missingness flags before median filling, and one-hot encode the categorical content type. The label is observed from `trend_direction`, but neither that field nor `trend_pct` enters the honest vector.

In [2]:
from pathlib import Path

import numpy as np
import pandas as pd
from sklearn.compose import ColumnTransformer
from sklearn.impute import SimpleImputer
from sklearn.linear_model import LogisticRegression
from sklearn.metrics import roc_auc_score
from sklearn.model_selection import GroupShuffleSplit
from sklearn.pipeline import Pipeline
from sklearn.preprocessing import OneHotEncoder, StandardScaler


def find_dataset() -> Path:
    candidates = [
        Path.cwd() / "data" / "raw" / "content_refresh_anonymized.csv",
        Path.cwd().parent / "data" / "raw" / "content_refresh_anonymized.csv",
        Path.cwd().parent.parent / "data" / "raw" / "content_refresh_anonymized.csv",
        Path("data/raw/content_refresh_anonymized.csv"),
    ]
    for candidate in candidates:
        if candidate.exists():
            return candidate
    raise FileNotFoundError("Could not find the starter dataset.")


df = pd.read_csv(find_dataset())
y = df["trend_direction"].fillna("").str.lower().eq("down").astype(int)

SAFE_NUMERIC_FEATURES = [
    "impressions_90d",
    "clicks_90d",
    "sessions_90d",
    "avg_position",
    "days_since_last_update",
]
SAFE_CATEGORICAL_FEATURES = ["content_type"]

X = df[SAFE_NUMERIC_FEATURES + SAFE_CATEGORICAL_FEATURES].copy()
for column in SAFE_NUMERIC_FEATURES:
    X[f"has_{column}"] = X[column].isna().astype(int)

numeric_features = SAFE_NUMERIC_FEATURES + [f"has_{column}" for column in SAFE_NUMERIC_FEATURES]
preprocessor = ColumnTransformer(
    transformers=[
        ("numeric", Pipeline([
            ("imputer", SimpleImputer(strategy="median")),
            ("scale", StandardScaler()),
        ]), numeric_features),
        ("categorical", Pipeline([
            ("imputer", SimpleImputer(strategy="most_frequent")),
            ("onehot", OneHotEncoder(handle_unknown="ignore")),
        ]), SAFE_CATEGORICAL_FEATURES),
    ]
)

clean_model = Pipeline([
    ("preprocess", preprocessor),
    ("model", LogisticRegression(max_iter=1000, random_state=42)),
])

print("Rows:", len(X))
print("Safe feature columns:", SAFE_NUMERIC_FEATURES + SAFE_CATEGORICAL_FEATURES)
print("Feature matrix fields after missingness flags:", len(X.columns))

Rows: 30000
Safe feature columns: ['impressions_90d', 'clicks_90d', 'sessions_90d', 'avg_position', 'days_since_last_update', 'content_type']
Feature matrix fields after missingness flags: 11


## 2. Feature notes (meaning, missing, categorical, available-when?)

- `impressions_90d`, `clicks_90d`, and `sessions_90d`: trailing activity totals known at the snapshot; numeric missing values use the training median.
- `avg_position`: average search position; zero is treated as missing because the data dictionary defines zero as no position data; a missingness flag is retained before imputation.
- `days_since_last_update`: content freshness signal known at scoring time; missing values use the training median with a missingness flag.
- `content_type`: page category known before scoring; missing categories use the training mode and then one-hot encoding handles categories.

All six base signals are available before the decision moment. The model is evaluated with a client-grouped holdout so pages from the same client do not cross the split.

In [3]:
X.loc[X["avg_position"] == 0, "avg_position"] = np.nan

assert set(SAFE_NUMERIC_FEATURES + SAFE_CATEGORICAL_FEATURES).issubset(X.columns)
assert X["avg_position"].isna().sum() > 0
assert all(column.startswith("has_") for column in X.columns if column.startswith("has_"))

print("avg_position values treated as missing:", int(X["avg_position"].isna().sum()))
print("Categorical features:", SAFE_CATEGORICAL_FEATURES)
print("Missingness flags:", [column for column in X.columns if column.startswith("has_")])
print("Availability check: every selected field is a pre-decision snapshot signal.")

avg_position values treated as missing: 1205
Categorical features: ['content_type']
Missingness flags: ['has_impressions_90d', 'has_clicks_90d', 'has_sessions_90d', 'has_avg_position', 'has_days_since_last_update']
Availability check: every selected field is a pre-decision snapshot signal.


## 3. The leakage hunt

I test the same client-grouped holdout twice. The honest model uses only pre-decision signals. The deliberately leaky model adds `trend_pct`, which is computed from the same trend comparison that creates the decline label. A sharp score jump is evidence of leakage, not model quality. The leaky column is removed immediately after the demonstration.

In [4]:
def precision_at_k(y_true, scores, k=50):
    result = pd.DataFrame({"label": y_true, "score": scores})
    return float(result.nlargest(k, "score")["label"].mean())


def make_model(numeric_columns, categorical_columns):
    transformer = ColumnTransformer(
        transformers=[
            ("numeric", Pipeline([
                ("imputer", SimpleImputer(strategy="median")),
                ("scale", StandardScaler()),
            ]), numeric_columns),
            ("categorical", Pipeline([
                ("imputer", SimpleImputer(strategy="most_frequent")),
                ("onehot", OneHotEncoder(handle_unknown="ignore")),
            ]), categorical_columns),
        ]
    )
    return Pipeline([
        ("preprocess", transformer),
        ("model", LogisticRegression(max_iter=1000, random_state=42)),
    ])


groups = df["client_id"]
splitter = GroupShuffleSplit(n_splits=1, test_size=0.2, random_state=42)
train_idx, test_idx = next(splitter.split(X, y, groups=groups))

clean_model.fit(X.iloc[train_idx], y.iloc[train_idx])
clean_scores = clean_model.predict_proba(X.iloc[test_idx])[:, 1]
clean_auc = roc_auc_score(y.iloc[test_idx], clean_scores)
clean_precision = precision_at_k(y.iloc[test_idx].to_numpy(), clean_scores)

X_leaky = X.copy()
X_leaky["trend_pct"] = df["trend_pct"]
leaky_model = make_model(numeric_features + ["trend_pct"], SAFE_CATEGORICAL_FEATURES)
leaky_model.fit(X_leaky.iloc[train_idx], y.iloc[train_idx])
leaky_scores = leaky_model.predict_proba(X_leaky.iloc[test_idx])[:, 1]
leaky_auc = roc_auc_score(y.iloc[test_idx], leaky_scores)
leaky_precision = precision_at_k(y.iloc[test_idx].to_numpy(), leaky_scores)

base_rate = float(y.iloc[test_idx].mean())
print("Client-grouped holdout clients:", groups.iloc[test_idx].nunique())
print("Base rate:", round(base_rate, 3))
print("Honest model ROC-AUC:", round(clean_auc, 3), "Precision@50:", round(clean_precision, 3))
print("Leaky model ROC-AUC:", round(leaky_auc, 3), "Precision@50:", round(leaky_precision, 3))
print("Leakage jump in ROC-AUC:", round(leaky_auc - clean_auc, 3))

assert "trend_pct" not in SAFE_NUMERIC_FEATURES
assert leaky_auc > clean_auc
X_leaky = X_leaky.drop(columns=["trend_pct"])
print("After the test, trend_pct removed from the retained feature frame.")

Client-grouped holdout clients: 7
Base rate: 0.511
Honest model ROC-AUC: 0.55 Precision@50: 0.68
Leaky model ROC-AUC: 0.951 Precision@50: 1.0
Leakage jump in ROC-AUC: 0.401
After the test, trend_pct removed from the retained feature frame.


## 4. What I excluded and why

- `trend_direction`: excluded because it directly defines the observed decline label.
- `trend_pct`: excluded because it is the numeric source used to derive `trend_direction`; the leakage test above confirms the score becomes misleadingly strong when it is added.
- `content_id` and `client_id`: retained only for row identity, reporting, and grouped splitting; pseudonymous IDs are not predictive features.
- `provider_used` and `model_used`: excluded because they describe the generation system rather than the page's pre-decision search opportunity, and may encode product/process decisions.
- Any post-snapshot or future-window field: excluded because it would not exist when the reviewer needs the ranking.

In [5]:
EXCLUDED_COLUMNS = [
    "trend_direction",
    "trend_pct",
    "content_id",
    "client_id",
    "provider_used",
    "model_used",
]

assert set(EXCLUDED_COLUMNS).isdisjoint(SAFE_NUMERIC_FEATURES + SAFE_CATEGORICAL_FEATURES)
assert "trend_pct" not in X_leaky.columns
assert "trend_direction" not in X.columns

print("Excluded columns:", EXCLUDED_COLUMNS)
print("Retained predictor columns:", list(X.columns))
print("Leakage and identifier audit passed.")

Excluded columns: ['trend_direction', 'trend_pct', 'content_id', 'client_id', 'provider_used', 'model_used']
Retained predictor columns: ['impressions_90d', 'clicks_90d', 'sessions_90d', 'avg_position', 'days_since_last_update', 'content_type', 'has_impressions_90d', 'has_clicks_90d', 'has_sessions_90d', 'has_avg_position', 'has_days_since_last_update']
Leakage and identifier audit passed.


## Self-check

- [x] Feature vector built with numeric, categorical, and missingness handling.
- [x] Feature meanings, missing-value treatment, categorical handling, and timing are documented.
- [x] Leakage attack compares clean and deliberately leaky models using a client-grouped holdout and prints the base rate.
- [x] Excluded fields have reasons, and the leaky field is removed after the demonstration.
- [ ] Run the notebook top to bottom, then commit it under `work/notebooks/`.